In [1]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "all" # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (1607, 41)


In [2]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

print("Features:")
print(X.columns.tolist())

Targets:
['cab']
Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


## X (features) as indices only

In [3]:
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())


X = X.drop(
        columns=AS7343_CHANNELS,
        errors="ignore",
    ).copy()

features = X.columns.tolist()

print("Features:")
print(features)

print(X.shape)

df = pd.concat([X, y], axis = 1).copy()
print(df.shape)


Features:
['ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']
(1607, 28)
(1607, 29)


## Hyperspectra up to 1000nm only

In [4]:
if spectral_length == "1000":
    print(X.shape)
    X_1000 = X.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y.shape)

    df = pd.concat([X_1000, y], axis = 1).copy()
    print(df.shape)

In [5]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every feature and every target
# --------------------------------------------------

for feature in features:

    print("\n" + "=" * 60)
    print(f"FEATURE: {feature}")
    print("=" * 60)

    X, y = preprocessing.split_dataset_into_X_y(df, target) # here "df" has multiple indices
    
    X_feature = X[[feature]].copy() # pick only one feature

    df_feature = pd.concat([X_feature, y], axis = 1).copy() # build new "df" with only one feature

    #check for duplicates
    df_feature = preprocessing.drop_duplicates(
        df_feature,
        target
    ).copy()

    # if df_feature changed, do this - here default
    X_feature, y = preprocessing.split_dataset_into_X_y(df_feature, target)

    X_feature = X_feature.copy()
    y = y.copy()

    # Train / validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_feature,
        y,
        test_size=0.20,
        random_state=42,
    )

    # Create a fresh model for every target
    model_xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=1.0,

        tree_method="hist",

        objective="reg:squarederror",
        eval_metric="rmse",

        n_jobs=-1,
        random_state=42,
    )

    # Train
    pipeline_xgb, metrics = train_model(
        model_xgb,
        X_train,
        y_train,
        X_val,
        y_val,
        scale=False,
    )

    metrics["Samples"] = len(df_feature)
    results[feature] = metrics


FEATURE: ci_red_edge_as7343
(Before) Shape of dataframe: (1607, 2)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 2)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 2)

FEATURE: ci_green_as7343
(Before) Shape of dataframe: (1607, 2)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 2)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 2)

FEATURE: gm_nd_as7343
(Before) Shape of dataframe: (1607, 2)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 2)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 2)

FEATURE: red_edge_ratio_as7343
(Before) Shape of dataframe: (1607, 2)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 2)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) 

In [6]:
# --------------------------------------------------
# Create results table
# --------------------------------------------------

results_df = pd.DataFrame(results)
print("\nXGBoost Results for each target")
print(results_df)


XGBoost Results for each target
                  ci_red_edge_as7343  ci_green_as7343  gm_nd_as7343  \
MAE (train)                 8.380944         8.913489      8.563727   
MAPE (train) [%]           27.769256        29.535365     28.404400   
MAE (test)                 10.215053        11.172676     11.191167   
MAPE (test) [%]            32.762247        36.639708     35.664293   
R² (test)                   0.164651        -0.010127      0.007164   
Samples                  1606.000000      1606.000000   1606.000000   

                  red_edge_ratio_as7343  datt2_as7343  datt1_as7343  \
MAE (train)                    8.563727      8.281262      9.370646   
MAPE (train) [%]              28.404400     27.206910     31.020525   
MAE (test)                    11.191167     10.564211     10.870595   
MAPE (test) [%]               35.664293     33.569980     36.222970   
R² (test)                      0.007164      0.084788      0.095960   
Samples                     1606.000000   1

In [7]:
results_df.head()

,ci_red_edge_as7343,ci_green_as7343,gm_nd_as7343,red_edge_ratio_as7343,datt2_as7343,datt1_as7343,mcari_as7343,tcari_as7343,osavi_as7343,mcari_osavi_as7343,...,sipi_as7343,psri_as7343,red_green_nd,red_edge_green_nd,nir_green_nd,red_edge_slope_proxy,green_red_ratio,rededge_green_ratio,nir_fy_ratio,nir_fxl_ratio
MAE (train),8.380944,8.913489,8.563727,8.563727,8.281262,9.370646,10.131190,9.142386,9.315732,10.212543,...,7.387018,8.543268,9.600825,9.263215,8.913489,10.415341,9.659458,9.263215,9.099544,9.224484
MAPE (train) [%],27.769256,29.535365,28.404400,28.404400,27.206910,31.020525,33.481929,30.374074,30.605474,34.116474,...,23.702191,28.111202,31.353813,31.064171,29.535365,34.893152,31.552467,31.064171,30.257282,30.606472
MAE (test),10.215053,11.172676,11.191167,11.191167,10.564211,10.870595,12.348861,10.661366,11.180490,12.394463,...,9.713351,10.492311,10.926979,11.426886,11.172676,12.706296,10.856744,11.426886,11.296205,11.298933
MAPE (test) [%],32.762247,36.639708,35.664293,35.664293,33.569980,36.222970,41.221163,35.578269,36.355382,41.407576,...,30.262074,34.343874,35.166395,37.501284,36.639708,41.963330,34.750324,37.501284,37.255496,36.943415
R² (test),0.164651,-0.010127,0.007164,0.007164,0.084788,0.095960,-0.089529,0.072535,0.043057,-0.085481,...,0.215260,0.138476,0.043246,-0.022487,-0.010127,-0.165279,0.027438,-0.022487,-0.028728,0.017404


In [8]:
results_df.iloc[-2, :].sort_values(ascending=False)

mndvi_as7343             0.236910
sipi_as7343              0.215260
ci_red_edge_as7343       0.164651
psri_as7343              0.138476
datt1_as7343             0.095960
ndvi_as7343              0.084788
sr_as7343                0.084788
datt2_as7343             0.084788
tcari_as7343             0.072535
tcari_osavi_as7343       0.063903
red_green_nd             0.043246
osavi_as7343             0.043057
green_red_ratio          0.027438
nir_fxl_ratio            0.017404
red_edge_ratio_as7343    0.007164
gm_nd_as7343             0.007164
ci_green_as7343         -0.010127
green_sr_as7343         -0.010127
gndvi_as7343            -0.010127
nir_green_nd            -0.010127
red_edge_green_nd       -0.022487
rededge_green_ratio     -0.022487
nir_fy_ratio            -0.028728
evi2_as7343             -0.053476
mcari_osavi_as7343      -0.085481
mcari_as7343            -0.089529
rdvi_as7343             -0.091251
red_edge_slope_proxy    -0.165279
Name: R² (test), dtype: float64

In [9]:
top_4_indices = (
    results_df.iloc[-2, :]
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

print(top_4_indices)

['mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343']


In [10]:
# file_name = f"xgboost_results_{target}_{spectral_length}-{indices}-only.csv"
# output_path = config.REPORTS_DIR / file_name

# results_df.to_csv(
#     output_path,
#     index=True,
# )

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets